In [ ]:
def sensFunc(rs,specInd,EOparams):
    degInvest = EOparams['degInvest']
    Kms = EOparams['Kms']
    Vmaxs = EOparams['Vmaxs']
    
    if specInd == 0: #Indicates degrader species
        return Vmaxs[0,0]*(1-degInvest[0])*(rs[0]/(rs[0] + Kms[0,0])) + Vmaxs[0,1]*(rs[1]/(rs[1] + Kms[0,1]))
    elif specInd == 1: #Indicates cross-feeder species (level 1)
        return Vmaxs[1,1]*(1-degInvest[1])*(rs[1]/(rs[1] + Kms[1,1]))

#Gradient of the sensitivity function in the resource space
def gradSensFunc(rs,specInd,EOparams):
    degInvest = EOparams['degInvest']
    Kms = EOparams['Kms']
    Vmaxs = EOparams['Vmaxs']
    
    sensGrad = np.zeros(rs.shape)
    
    if specInd == 0:
        sensGrad[0] = Vmaxs[0,0]*(1-degInvest[0])*Kms[0,0]/(rs[0]+Kms[0,0])**2
        sensGrad[1] = Vmaxs[0,1]*Kms[0,1]/(rs[1]+Kms[0,1])**2
    elif specInd == 1: 
        sensGrad[1] = Vmaxs[1,1]*(1-degInvest[1])*Kms[1,1]/(rs[1]+Kms[1,1])**2
    
    return sensGrad

#Impact function for a single species
def impactFunc(rs,specInd,EOparams):
    degInvest = EOparams['degInvest']
    Kms = EOparams['Kms']
    Vmaxs = EOparams['Vmaxs']
    Ys = EOparams['Ys']
    
    impact = np.zeros(rs.shape)
    
    if specInd == 0:
        impact[0] = -(1/Ys[0,0])*Vmaxs[0,0]*rs[0]/(rs[0] + Kms[0,0])
        impact[1] = -(1/Ys[0,1])*Vmaxs[0,1]*rs[1]/(rs[1] + Kms[0,1]) + degInvest[0]*Vmaxs[0,0]*rs[0]/(rs[0] + Kms[0,0])
    elif specInd == 1:
        impact[1] = -(1/Ys[1,1])*Vmaxs[1,1]*rs[1]/(rs[1] + Kms[1,1])
        
    return impact

def instInteract(rs,alphInd,betInd,EOparams):
    return(np.dot(impactFunc(rs,betInd,EOparams),gradSensFunc(rs,alphInd,EOparams)))

def intrinsicGR(rs,alphInd,EOparams,sysParams):
    return(np.dot(gradSensFunc(rs,alphInd,EOparams),sysParams['D']*(sysParams['rIn']-rs)))

In [ ]:
Kms = np.array([[1,2.5],[-1,2.5]]) #Indexed by species (dim 1), resource (dim 2). Negative values indicate parameter not used
Vmaxs = np.array([[1,0.1],[-1,1]]) #Indexed by species (dim 1), resource (dim 2)
Ys = np.array([[1,1],[-1,1]]) #Indexed by species (dim 1), resource (dim 2)
degInvest = np.array([0.65,-1]) #Only relevant for degraders - fraction of polymer that gets converted to monomer rather than degrader biomass

EOparams = {'Kms':Kms,'Vmaxs':Vmaxs,'Ys':Ys,'degInvest':degInvest}